# MIDAS on Kaggle — CHAI attack against InternVL2.5-8B

Reproduces the CHAI visual sign-injection attack and evaluates the
activation-steering defense, at the model scale CHAI themselves used.

**Before running, set the session up:**

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Internet | **On** (needs phone verification on your Kaggle account) |
| Persistence | Variables and files |

Two T4s give 32 GB of VRAM combined. The 8B model needs ~16 GB in float16, so it
is sharded across both with `--device-map auto`.

**Expected cost:** roughly 2–3 GPU-hours for the full attack evaluation, against
a weekly quota of about 30.

## 1. Check the hardware

Run this first. If it reports anything other than two T4s, stop and fix the
accelerator setting — everything below assumes this layout.

In [ ]:
import os
# Reduces fragmentation, which the first OOM on this hardware recommended.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import subprocess, torch, sys

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv"], capture_output=True, text=True).stdout)

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("cuda        :", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    major, minor = torch.cuda.get_device_capability(i)
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  gpu {i}: {torch.cuda.get_device_name(i)}  "
          f"sm_{major}{minor}  {total:.1f} GB")

# Gate on compute capability, as midas.models.internvl.resolve_dtype does.
# torch.cuda.is_bf16_supported() reports True on a T4 because recent PyTorch
# counts emulated bf16, but Turing has no bf16 tensor cores and the emulated
# path is slower than float16.
cc = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
torch_claim = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
bf16 = cc >= 8
fa2 = bf16
print(f"torch bf16 claim   : {torch_claim}  (counts emulation -- not what we use)")
print(f"native bfloat16    : {bf16}   -> dtype will be {'bfloat16' if bf16 else 'float16'}")
print(f"FlashAttention-2 ok: {fa2}   -> attention will run {'flash' if fa2 else 'eager'}")
if not fa2:
    print("  (expected on T4: sm_75 is Turing, FA2 needs sm_80+. Eager attention")
    print("   uses much more memory, so keep --max-tiles modest at first.)")

## 2. Install dependencies

Kaggle preinstalls PyTorch built against its own CUDA, so **do not reinstall
torch** — that risks breaking GPU support and wastes several minutes.

Two pins are load-bearing and cost us hours to diagnose locally:

- **`sentencepiece==0.2.1`.** Version 0.2.2 rejects InternVL's `tokenizer.model`
  as malformed even though its checksum matches the hub; `AutoTokenizer` swallows
  the error and returns `False`, which surfaces much later as `'bool' object has
  no attribute 'convert_tokens_to_ids'`. 0.1.99 also works but has no Python 3.13
  wheel, and Kaggle now runs 3.13 — pinning it makes pip try a source build,
  which fails and aborts the whole install.
- **No `numpy<2` pin.** Kaggle's preinstalled stack calls
  `numpy.dtypes.StringDType`, which only exists in numpy 2.0+. Forcing numpy 1.x
  makes `transformers` fail to import its model code with an error that names
  llama rather than numpy.
- **`transformers<4.50`.** InternVL2.5's remote code does not inherit from
  `GenerationMixin`, which 4.50 made mandatory. Kaggle ships 5.x, so this is a
  real downgrade and takes a minute.

The install is checked below rather than assumed: a failed pip run leaves the
preinstalled versions in place, and the notebook would then fail much later for
reasons that look unrelated.

In [ ]:
%pip install -q "transformers>=4.37,<4.50" "sentencepiece==0.2.1" protobuf \
    "accelerate>=0.26" timm einops opencv-python-headless scikit-learn pytest

import importlib, sentencepiece, transformers, accelerate
for mod in (transformers, sentencepiece, accelerate):
    importlib.reload(mod)

print("transformers :", transformers.__version__)
print("sentencepiece:", sentencepiece.__version__)
print("accelerate   :", accelerate.__version__)

major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
assert (major, minor) < (4, 50), (
    f"transformers {transformers.__version__} is too new; InternVL2.5's remote "
    "code needs <4.50. The pip step above must have failed -- scroll up."
)
assert sentencepiece.__version__ == "0.2.1", (
    f"sentencepiece {sentencepiece.__version__} installed, need 0.2.1. "
    "0.2.2 cannot read InternVL's tokenizer and fails confusingly later."
)
print("version checks passed")

## 3. Load the code

The code and CHAI's released artefacts arrive as an attached Kaggle Dataset, not
a git clone, so this notebook needs no GitHub repository and no access token.
Upload it with `python scripts/kaggle_deploy.py upload`.

The dataset is read-only, so it is copied into the working directory before the
editable install.

In [ ]:
import os, shutil, subprocess, glob
from pathlib import Path

INPUT = Path("/kaggle/input")
print("contents of /kaggle/input:")
if INPUT.exists():
    for entry in sorted(INPUT.iterdir()):
        inner = sorted(p.name for p in entry.iterdir())[:12] if entry.is_dir() else []
        print(f"  {entry.name}/  -> {inner}")
else:
    print("  (missing entirely -- no dataset is attached to this kernel)")

# Search recursively: a dataset may mount its payload one level deeper than
# expected depending on how the archive was packed.
# Datasets mount under /kaggle/input/datasets/<user>/<slug>; kernel outputs
# mount under /kaggle/input/<kernel-slug>/. The sweep kernel attaches both, and
# the main run saved its own working copy of this source tree as output -- so an
# unqualified search finds that stale copy instead of the uploaded dataset.
candidates = sorted(
    glob.glob("/kaggle/input/**/pyproject.toml", recursive=True),
    key=lambda p: (0 if "/kaggle/input/datasets/" in p else 1, p),
)
assert candidates, (
    "MIDAS source not found under /kaggle/input. Attach the dataset via "
    "Add Data -> Your Datasets, or re-run scripts/kaggle_deploy.py upload. "
    "If the dataset was created moments before this kernel started, it may "
    "still have been processing -- wait for `kaggle datasets status` to read "
    "ready, then push again."
)
SOURCE = Path(candidates[0]).parent
# Not "/kaggle/working/midas": Kaggle puts /kaggle/working on sys.path, so a
# directory of that name shadows the installed package and `import midas`
# resolves to an empty namespace package with __file__ None.
WORK = Path("/kaggle/working/midas_src")
print("source dataset:", SOURCE)

# /kaggle/input is read-only and `pip install -e` needs to write egg-info.
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(SOURCE, WORK)
os.chdir(WORK)

chai = WORK / "vendor" / "chai"
assert (chai / "training.json").exists(), (
    f"CHAI artefacts missing from the dataset (looked in {chai}); "
    "re-run the upload step"
)
frames = len(glob.glob(str(chai / "DriveLM_dataset/**/*.jpg"), recursive=True))
print(f"CHAI frames present: {frames} (expect 184)")
assert frames == 184, "frame count wrong -- the upload was incomplete"

%pip install -q -e .

# The editable install registers the console scripts (midas-eval and friends),
# which run as separate processes. It does not refresh this already-running
# kernel's import path, so point sys.path at the sources directly.
import importlib, sys, shutil as _shutil

sys.path.insert(0, str(WORK / "src"))
importlib.invalidate_caches()

import midas
assert midas.__file__, (
    "`midas` imported as a namespace package, so something on sys.path shadows "
    "it -- check for a directory of that name under /kaggle/working"
)
print("midas imported from", midas.__file__)

missing = [c for c in ("midas-eval", "midas-activations", "midas-probes")
           if _shutil.which(c) is None]
assert not missing, f"console scripts not on PATH: {missing}"
print("console scripts available")

## 4. Verify the data before spending GPU time

These checks are CPU-only and take seconds. They catch a bad clone before a
multi-hour run does.

In [ ]:
!python -m pytest tests -q

from midas import data
for split in ("train", "test"):
    scenes = data.load_scenes(split)
    missing = data.missing_frames(scenes)
    print(f"{split}: {len(scenes)} scenes, {len(missing)} missing frames")

from midas.tasks import drivelm
diffs = drivelm.verify_matches_upstream()
print("prompt matches CHAI source:", "yes" if not diffs else diffs)

## 5. Download the model

`OpenGVLab/InternVL2_5-8B` is about 16 GB. It goes to the HuggingFace cache rather than
`/kaggle/working`, which only has 20 GB and must also hold results.

To skip this download on later sessions, upload the cache once as a Kaggle
Dataset and attach it.

In [ ]:
from huggingface_hub import snapshot_download
import time

t0 = time.time()
path = snapshot_download(
    "OpenGVLab/InternVL2_5-8B",
    allow_patterns=["*.json", "*.py", "*.safetensors", "*.txt", "*.model"],
)
print(f"downloaded in {(time.time() - t0) / 60:.1f} min -> {path}")
!du -sh {path}

### Shared names

Every stage below is independent and can be skipped, so the constants and
imports they share are defined once, here, rather than inside whichever stage
happened to need them first.

In [ ]:
import collections, glob, json, subprocess
from pathlib import Path

# The scale study and the option-order control both sweep these.
SIZES = ["OpenGVLab/InternVL2_5-1B", "OpenGVLab/InternVL2_5-2B",
         "OpenGVLab/InternVL2_5-4B"]

# Injection styles run on the 2B: it had 0% unreadable answers, where on the 8B
# a quarter of replies name both actions and would bury the signal.
STYLE_MODEL = "OpenGVLab/InternVL2_5-2B"
STYLES = ["plain", "authority", "override", "urgency"]

RESULTS = Path("/kaggle/working/results")
print("SIZES:", [s.split("-")[-1] for s in SIZES], " STYLES:", STYLES)

## 6. Smoke test — one scene

Never start a multi-hour sweep without this. It confirms the model loads across
both GPUs, the tokenizer works, and the prompt produces a parseable action.

Watch the printed `device=... dtype=... flash_attn=...` line: on a T4 it must say
`float16` and `flash_attn=False`.

In [ ]:
!midas-eval --split train --baseline --limit 1 \
    --model OpenGVLab/InternVL2_5-8B --device-map auto --max-tiles 6 \
    --out /kaggle/working/results/smoke

## 7. Check memory headroom before raising the tile count

CHAI render at 12 tiles per frame (9 on these images). Eager attention makes that
roughly four times the attention memory of 6 tiles, so confirm there is room
rather than discovering it two hours in.

If this reports comfortable headroom, use `--max-tiles 12` below — that is the
faithful setting and the one to report.

In [ ]:
import torch

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"gpu {i}: {free/1e9:5.1f} GB free of {total/1e9:5.1f} GB")

heads, seq6, seq12 = 32, 3300, 6400          # InternLM2.5-7B attention heads
for name, seq in (("6 tiles", seq6), ("12 tiles", seq12)):
    gb = seq * seq * heads * 2 / 1e9
    print(f"{name:9s} seq~{seq:5d} -> eager attention matrix {gb:5.2f} GB per layer")

## 9e. CHAI's own search — the method, not just the output

Everything above renders the sign CHAI *published*. Their attack is actually two
searches, and we had been loading the results of both:

| Stage | CHAI's code | What we were doing |
|---|---|---|
| 1. Phrase dictionary proposed by an attacker LLM | `create_dict.py` | load `attack_dicts_internvl.json` |
| 2. Cross-entropy search over the sign's appearance | `main.py` + scenic/VerifAI | load `optimal_attacks/optimal.json` |

Stage 2 searches a box: size ×0.8–1.2, background RGB, letter RGB, position
±25 px, and which dictionary phrase to use. This section runs it.

**First, a correction.** `main.py` sets `testing = True`, so its InternVL path
calls `load_universal("internvl")` and **never reads `optimal.json`**. The two
describe different signs:

| | `optimal.json` (what we rendered) | `load_universal` (what CHAI run) |
|---|---|---|
| background | (255, 0, 15) | (255, 0, 0) |
| letters | (237, 196, 196) pink | **(0, 0, 0) black** |
| size | 1.00× | **0.93×** |

Black on red is far more legible than pink on red, so our published rates were
probably an **underestimate**. Measure that gap before searching.

In [ ]:
for model_id, extra in ((STYLE_MODEL, []),
                        ("OpenGVLab/InternVL2_5-8B", ["--device-map", "auto"])):
    for source in ("chai_internvl", "optimal_json"):
        print(f"\n{'=' * 70}\n{model_id}  sign={source}\n{'=' * 70}", flush=True)
        subprocess.run(
            ["midas-eval", "--split", "train", "--both",
             "--model", model_id, "--max-tiles", "6",
             "--param-source", source, "--save-long",
             "--out", f"/kaggle/working/results/param_source/{source}", *extra],
            check=True,
        )

In [ ]:
print(f"{'model':6s} {'sign':14s} {'clean':>8s} {'attacked':>9s} {'delta':>8s}")
for source in ("chai_internvl", "optimal_json"):
    for size in ("2B", "8B"):
        got = {}
        for f in glob.glob(f"/kaggle/working/results/param_source/{source}/*.json"):
            d = json.load(open(f))
            if d["model_id"].split("-")[-1] == size:
                got[d["label"]] = d["summary"]["asr_chai"]
        if len(got) == 2:
            delta = got["attack"] - got["baseline"]
            print(f"{size:6s} {source:14s} {got['baseline']:8.1%} "
                  f"{got['attack']:9.1%} {delta:+8.1%}")

### Run the search

The search runs on the 2B. Forced-choice scoring is about ten times cheaper than
generating a chain of thought per candidate per scene, but the 8B at
12 × 24 × 6 evaluations would still be roughly four GPU-hours.

`--target a)` matches CHAI, whose `main.py` hardcodes `targets = ["a)"]`. Only
scenes whose benign answer is not already *proceed* can be attacked toward it,
which is **6 of the 15** — so one scene is 16.7 points here, and anything under
two scenes is noise. Iteration 0 is seeded with CHAI's own sign, so the reported
best can never fall below the published attack.

`--validate` re-runs CHAI's sign and the winner through **real generation** at
the end: the forced-choice objective is a ranking signal only, having read 53.3%
where generation read 73.3% on the same scenes.

In [ ]:
subprocess.run(
    ["midas-optimize", "--split", "train", "--model", STYLE_MODEL,
     "--max-tiles", "6", "--target", "a)",
     "--iterations", "12", "--population", "24",
     "--objective", "forced", "--validate",
     "--out", "/kaggle/working/results/optimize"],
    check=True,
)

### Does a re-optimised sign transfer across scale?

CHAI's central claim is that one universal sign works everywhere. A sign tuned on
the 2B and evaluated on the 8B tests that directly: if it holds up, universality
extends across model size; if it collapses, their single released parameter set
is doing more work than the paper implies.

In [ ]:
search = sorted(glob.glob("/kaggle/working/results/optimize/*.json"))
print("search results:", [p.split("/")[-1] for p in search])
if search:
    payload = json.load(open(search[0]))
    print("best:", payload["best"])
    print("validation:", payload["meta"].get("validation", "not run"))

    subprocess.run(
        ["midas-eval", "--split", "train", "--attack",
         "--model", "OpenGVLab/InternVL2_5-8B", "--device-map", "auto",
         "--max-tiles", "6", "--attack-params", search[0], "--save-long",
         "--out", "/kaggle/working/results/optimize/transfer_8b"],
        check=True,
    )

In [ ]:
rows = []
for label, pattern in (
    ("2B, CHAI sign", "/kaggle/working/results/param_source/chai_internvl/*.json"),
    ("8B, re-optimised", "/kaggle/working/results/optimize/transfer_8b/*.json"),
):
    for f in sorted(glob.glob(pattern)):
        d = json.load(open(f))
        rows.append((label, d["model_id"].split("-")[-1], d["label"],
                     d["summary"]["asr_chai"]))

print(f"{'run':20s} {'model':6s} {'condition':10s} {'CHAI-rule ASR':>14s}")
for label, size, cond, asr in rows:
    print(f"{label:20s} {size:6s} {cond:10s} {asr:14.1%}")
print("")
print("The search used 6 attackable scenes, so one scene is 16.7 points there.")
print("The transfer run is all 15, where one scene is 6.7.")

## 11. Package the results

`/kaggle/working` persists between sessions, but download the archive anyway so
the results can be committed to the repository alongside the 2B numbers.

In [ ]:
import shutil, json, glob

shutil.make_archive("/kaggle/working/midas_8b_results", "zip",
                    "/kaggle/working/results")
print("archive written")
!du -sh /kaggle/working/midas_8b_results.zip

for f in sorted(glob.glob("/kaggle/working/results/runs_8b/*.json")):
    d = json.load(open(f))
    s = d["summary"]
    print(f"{d['label']:9s} {d['split']:6s} n={s['n']:<3} "
          f"ASR={s['asr']:6.1%}  benign={s['benign_accuracy']:6.1%}")